In [ ]:
from TIEModel import TIEModel
import torch, math
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from TIEUtils import collator, TemporalDataset
from torch.utils.data import DataLoader
from globals import ID2LABEL_EVNER, ID2LABEL_EE, LABEL2ID_EVNER, LABEL2ID_EE

In [ ]:
label2id_ner = LABEL2ID_EVNER
id2label_ner = ID2LABEL_EVNER
label2id_ee = LABEL2ID_EE
id2label_ee = ID2LABEL_EE
cleandata_path = "D:\\GeoTKG\\cleandata\\tie\\"
def collate_fn(examples):
    return collator(examples, label2id_ner=label2id_ner, label2id_ee=label2id_ee)
train = TemporalDataset(cleandata_path + "train.json")
eval = TemporalDataset(cleandata_path + "eval.json")
train_loader = DataLoader(train, batch_size=16, shuffle=True, collate_fn=collate_fn)
eval_loader = DataLoader(eval, batch_size=16, shuffle=False, collate_fn=collate_fn)

In [ ]:
NUM_EPOCHS = 50
ENC_LR = 5e-5
NONENC_LR = 1e-3
WARMUP_EPOCHS = 51
LOGGING_EPOCHS = 10
BASE_ENC_MODEL = "roberta-base"
HEADS = 6
WEIGHT_DECAY = 0.01

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

model = TIEModel(base=BASE_ENC_MODEL, num_ner=len(label2id_ner), ee_labels=len(label2id_ee), heads=HEADS).to(device)

for p in model.enc.parameters(): p.requires_grad = False

optimizer = AdamW([
        {"params": [p for n,p in model.named_parameters() if n.startswith("enc.")], "lr": ENC_LR},
        {"params": [p for n,p in model.named_parameters() if not n.startswith("enc.")], "lr": NONENC_LR},
    ], weight_decay=0.01)

steps_per_epoch = len(train_loader)
num_train_steps = steps_per_epoch * NUM_EPOCHS
num_warmup_steps = steps_per_epoch * WARMUP_EPOCHS

sched = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_train_steps)

scaler = torch.amp.GradScaler(enabled=(device.type=='cuda'))

def make_optim(unfrozen: bool):
    groups = []
    if unfrozen:
        groups.append({"params": model.enc.parameters(), "lr": ENC_LR})
    else:
        # keep enc group empty or skip entirely; either is fine
        pass
    nonenc = [p for n, p in model.named_parameters() if not n.startswith("enc.")]
    groups.append({"params": nonenc, "lr": NONENC_LR})
    return AdamW(groups, weight_decay=WEIGHT_DECAY)

In [ ]:
global_step = 0
history = {#  Training
           "loss": [], "ner_loss":[], "ptr_loss":[], "ee_loss":[], "lr": [],
           #  Evaluation
           "ner_f1": [], "ptr_acc": [], "ee_f1": [], "eval_loss":[], "ner_eval_loss": [], "ptr_eval_loss": [], "ee_eval_loss": []}
for epoch in range(NUM_EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss_sum = 0
    ner_loss_sum = 0
    ptr_loss_sum = 0
    ee_loss_sum = 0
    for step, batch in enumerate(train_loader):
        batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}

        ctx = (torch.autocast(device_type='cuda', dtype=torch.float16))
        with ctx:
            out = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                ev_starts=batch["ev_starts"], ev_ends=batch["ev_ends"], ev_mask=batch["ev_mask"], e_sent_ids=batch["e_sent_ids"],
                ti_starts=batch["ti_starts"], ti_ends=batch["ti_ends"], ti_mask=batch["ti_mask"], t_sent_ids=batch["t_sent_ids"],
                ner_gold_labels=batch["ner_labels"],
                ev_ti_gold=batch["ev_ti_gold"],
                ee_rel_gold=batch["ee_triples"],
                ee_mask=batch["ee_mask"],
            )
            loss = out["loss"]
            loss_sum += loss.item()
            ner_loss_sum += out["ner_loss"].item()
            ptr_loss_sum += out["ptr_loss"].item()
            ee_loss_sum += out["ee_loss"].item()

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        prev_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        sched.step() if scaler.get_scale() <= prev_scale else None
        optimizer.zero_grad(set_to_none=True)
        global_step += 1

    print(f"Epoch {epoch+1} done. Avg Loss: {loss_sum / steps_per_epoch:.4f}")
 
    # ---- unfreeze after warmup epochs ----
    if epoch + 1 == WARMUP_EPOCHS:
        for p in model.enc.parameters():
            p.requires_grad = True
        # rebuild optimizer & scheduler for the remaining steps
        optimizer = make_optim(unfrozen=True)
        remaining_steps = steps_per_epoch * (NUM_EPOCHS - (epoch + 1))
        warmup_rem = 0
        sched = get_cosine_schedule_with_warmup(optimizer, warmup_rem, remaining_steps)

    # ---- validation ----
    if (epoch + 1) % LOGGING_EPOCHS == 0:
        model.eval()
        with torch.no_grad():
            batch_evaluation = model.evaluate_dataloader(eval_loader, id2label_ner, id2label_ee)
            history["ner_f1"].append(batch_evaluation["ner_f1"])
            history["ptr_acc"].append(batch_evaluation["ptr_acc"])
            history["ee_f1"].append(batch_evaluation["ee_f1"])
            history["eval_loss"].append(batch_evaluation["eval_loss"])
            history["ner_eval_loss"].append(batch_evaluation["ner_loss"])
            history["ptr_eval_loss"].append(batch_evaluation["ptr_loss"])
            history["ee_eval_loss"].append(batch_evaluation["ee_loss"])
            history["loss"].append(loss_sum / steps_per_epoch)
            history["ner_loss"].append(ner_loss_sum / steps_per_epoch)
            history["ptr_loss"].append(ptr_loss_sum / steps_per_epoch)
            history["ee_loss"].append(ee_loss_sum / steps_per_epoch)
        model.save(f"results/tie_model/tie_model_epoch{epoch+1}.pt")
        print(f"EPOCH{epoch+1} \n NER F1={batch_evaluation['ner_f1']:.4f},  PTR={batch_evaluation['ptr_acc']:.4f},  EE F1={batch_evaluation['ee_f1']:.4f} \n Train Loss={loss_sum / steps_per_epoch:.4f},  Eval Loss={batch_evaluation['eval_loss']:.4f} \nNER Eval Loss={batch_evaluation['ner_loss']:.4f}, NER Train Loss={ner_loss_sum / steps_per_epoch:.4f},\nPTR Eval Loss={batch_evaluation['ptr_loss']:.4f}, PTR Train Loss={ptr_loss_sum / steps_per_epoch:.4f}\nEE Eval Loss={batch_evaluation['ee_loss']:.4f},EE Train Loss={ee_loss_sum / steps_per_epoch:.4f}")

In [ ]:
import matplotlib.pyplot as plt

x_epochs = list(range(10, 51, 10))
x_steps = list(range(1, global_step + 1, LOGGING_STEPS))

plt.plot( history["ner_f1"], label="NER F1")
plt.plot( history["ptr_acc"], label="Pointer Accuracy")
plt.plot( history["ee_f1"], label="EE F1")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Evaluation Metrics Over Time")
plt.legend()
plt.show()

In [ ]:
plt.plot(history["lr"], label="LR")
plt.xlabel("Step")
plt.ylabel("LR")
plt.title("Learning Rate Schedule")
plt.legend()
plt.show()

In [ ]:
plt.plot( history["loss"], label="Loss")
plt.plot( history["ner_loss"], label="NER Loss")
plt.plot( history["ee_loss"], label="EE Loss")
plt.plot( history["ptr_loss"], label="PTR Loss")
plt.legend()
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.show()

In [ ]:
plt.plot( history["eval_loss"], label="Loss")
plt.plot( history["ner_eval_loss"], label="NER Loss")
plt.plot( history["ee_eval_loss"], label="EE Loss")
plt.plot( history["ptr_eval_loss"], label="PTR Loss")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Evaluation Loss")
plt.show()

In [ ]:
import json
with open("results/tie_model/history.json", "w") as f:
    json.dump(history, f, indent=2)

In [ ]:
plt.plot( history["eval_loss"], label="Eval Loss")
plt.plot( history["loss"], label="Train Loss")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Compare Loss")
plt.show()